# Coding 4: Construct the exact annual observation model

QIDIAN XING  
MICHAEL SUHERMAN  
GUS GUENTHER  
ENDI ZONANA

MGMTMFE 431 Homework 1. Six-notebook organization: October 5, 2026.

**Formal task:** derive the 24-dimensional model for rolling twelve-month sums, give every matrix and the joint stationary prior, and construct/check them directly in code.

This notebook is independent: start its kernel in this directory and use Restart Kernel → Run All. No other notebook or project module is required. AI assistance is disclosed in the separate submission record.

## Parameters and fixed seeds

Monthly rates are in percentage points. Q and H are variances. Experimental settings are unchanged from the verified project.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import hashlib
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from numba import njit
from scipy.optimize import minimize
from IPython.display import display, HTML

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.2, 'figure.dpi': 120,
})
COLORS = {'monthly': '#1e4e79', 'annual': '#9b4b2b',
          'benchmark': '#52665f', 'data': '#454545'}

In [2]:
TRUE_PARAMETERS = {'mu': 0.2, 'phi': 0.9, 'Q': 0.04, 'H': 0.01}

## Rolling sums, lag blocks and the observation equation
$$z_t=\sum_{j=0}^{11}y_{t-j}=12\mu+\sum_{j=0}^{11}\alpha_{t-j}+\sum_{j=0}^{11}\eta_{t-j},$$
$$\alpha_t^{aug}=(\alpha_t,\ldots,\alpha_{t-11},\eta_t,\ldots,\eta_{t-11})'\in\mathbb R^{24}.$$
The annual-only conditional mean is $a^{aug}_{t|s}=E(\alpha^{aug}_t\mid z_{12:s};\theta)$;
its covariance is $P^{aug}_{t|s}$. The original monthly $a_{t|s}$ conditions on $y_{1:s}$.
$$z_t=d^{aug}+Z^{aug}\alpha_t^{aug},\qquad
\alpha_{t+1}^{aug}=c^{aug}+T^{aug}\alpha_t^{aug}+R^{aug}e_{t+1}.$$
Every system matrix, including its dimension, is specified below:
$$d^{aug}=12\mu\ (1),\quad Z^{aug}=\mathbf1_{1\times24},\quad
H^{aug}=0\ (1\times1),\quad c^{aug}=\mathbf0_{24},$$
$$T^{aug}=\begin{pmatrix}A_\phi&0\\0&A_0\end{pmatrix}\ (24\times24),\quad
e_{t+1}=(\epsilon_{t+1},\eta_{t+1})'\sim N(0,Q^{aug}),\quad Q^{aug}=\operatorname{diag}(Q,H)\ (2\times2).$$
For $i,j=1,\ldots,12$, define all entries of the two $12\times12$ blocks:
$$ (A_\phi)_{ij}=\phi\mathbf1_{i=j=1}+\mathbf1_{i=j+1},\qquad
(A_0)_{ij}=\mathbf1_{i=j+1}.$$
The loading $R^{aug}$ is $24\times2$: $R^{aug}_{1,1}=R^{aug}_{13,2}=1$ and every
other entry is zero. No extra observation shock is added outside the augmented state.

## Joint stationary prior before the first annual observation
$$\alpha_{12}^{aug}\sim N(\mathbf0_{24},P_0^{aug}),\quad
a^{aug}_{12|11}=\mathbf0_{24},\quad
P_0^{aug}=\begin{pmatrix}\Sigma_\alpha&0\\0&H I_{12}\end{pmatrix},$$
$$ (\Sigma_\alpha)_{ij}=\frac{Q}{1-\phi^2}\phi^{|i-j|},\quad i,j=1,\ldots,12.$$
This unconditional joint prior is taken before seeing $z_{12}$, with no monthly-data
initialization. It solves $P_0^{aug}=T^{aug}P_0^{aug}(T^{aug})'+R^{aug}Q^{aug}(R^{aug})'$.

Adjacent sums share monthly observation shocks. If $u_t=\sum_{j=0}^{11}\eta_{t-j}$,
then $\operatorname{Cov}(u_t,u_{t-k})=(12-k)H$ for $0\leq k\leq11$ and zero afterward.
$H^{aug}=0$ therefore does **not** imply $H=0$; the positive monthly H is carried by
the noise lag block. A stable additive seasonal pattern with a zero twelve-month sum
cancels; changing patterns need not. Summing is exact for log inflation and approximates
compounding for simple monthly rates. We use the required raw monthly series without
converting it to log rates. The 936 months yield 925 monthly-updated annual observations.

In [3]:
@dataclass(frozen=True)
class Parameters:
    mu: float
    phi: float
    Q: float
    H: float

    def __post_init__(self):
        if not (abs(self.phi) < 1 and self.Q > 0 and self.H > 0):
            raise ValueError("Require |phi|<1, Q>0 and H>0")

    def vector(self):
        return np.array([self.mu, self.phi, self.Q, self.H])

    def reported(self):
        return np.array([self.mu, self.phi, np.sqrt(self.Q), np.sqrt(self.H)])

In [4]:
def augmented_matrices(p):
    A_phi = np.zeros((12, 12))
    A_zero = np.zeros((12, 12))
    A_phi[0, 0] = p.phi
    for i in range(1, 12):
        A_phi[i, i-1] = A_zero[i, i-1] = 1
    T = np.zeros((24, 24))
    T[:12, :12], T[12:, 12:] = A_phi, A_zero
    R = np.zeros((24, 2))
    R[0, 0], R[12, 1] = 1, 1
    P0 = np.zeros((24, 24))
    lag = np.abs(np.arange(12)[:, None] - np.arange(12)[None, :])
    P0[:12, :12] = p.Q/(1-p.phi**2)*p.phi**lag
    P0[12:, 12:] = p.H*np.eye(12)
    return dict(A_phi=A_phi, A_zero=A_zero, Z=np.ones((1, 24)),
                d=np.array([12*p.mu]), H_obs=np.zeros((1, 1)), T=T,
                c=np.zeros(24), R=R, Q_aug=np.diag([p.Q, p.H]),
                a0=np.zeros(24), P0=P0)

## Construct and display every system matrix

These arrays are constructed from the model equations, not loaded from old matrix CSVs.

In [5]:
parameters = Parameters(**TRUE_PARAMETERS)
matrices = augmented_matrices(parameters)
display(pd.DataFrame([{'Matrix': name, 'Shape': str(value.shape)} for name,value in matrices.items()]))
for name, value in matrices.items():
    display(HTML(f'<h4>{name}: {value.shape}</h4><div style="overflow:auto">' +
                 pd.DataFrame(np.atleast_2d(value)).to_html(index=False, header=False,
                     float_format=lambda v: f'{v:.8g}') + '</div>'))

,Matrix,Shape
0,A_phi,"(12, 12)"
1,A_zero,"(12, 12)"
2,Z,"(1, 24)"
3,d,"(1,)"
4,H_obs,"(1, 1)"
5,T,"(24, 24)"
6,c,"(24,)"
7,R,"(24, 2)"
8,Q_aug,"(2, 2)"
9,a0,"(24,)"


0.9,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
0,1,0,0,0,0,0,0,0,0,0,0
0,0,1,0,0,0,0,0,0,0,0,0
0,0,0,1,0,0,0,0,0,0,0,0
0,0,0,0,1,0,0,0,0,0,0,0
0,0,0,0,0,1,0,0,0,0,0,0
0,0,0,0,0,0,1,0,0,0,0,0
0,0,0,0,0,0,0,1,0,0,0,0
0,0,0,0,0,0,0,0,1,0,0,0
0,0,0,0,0,0,0,0,0,1,0,0


0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
0,1,0,0,0,0,0,0,0,0,0,0
0,0,1,0,0,0,0,0,0,0,0,0
0,0,0,1,0,0,0,0,0,0,0,0
0,0,0,0,1,0,0,0,0,0,0,0
0,0,0,0,0,1,0,0,0,0,0,0
0,0,0,0,0,0,1,0,0,0,0,0
0,0,0,0,0,0,0,1,0,0,0,0
0,0,0,0,0,0,0,0,1,0,0,0
0,0,0,0,0,0,0,0,0,1,0,0


1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


2.4


...

0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


1,0
0,0
0,0
0,0
0,0
0,0
0,0
0,0
0,0
0,0
0,0


0.04,0
0,0.01


0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0.12431368,0.11188232,0.10069408,0.090624676,0.081562208,0.073405987,0.066065389,0,0,0,0,0,0,0,0,0,0,0,0
0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0.12431368,0.11188232,0.10069408,0.090624676,0.081562208,0.073405987,0,0,0,0,0,0,0,0,0,0,0,0
0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0.12431368,0.11188232,0.10069408,0.090624676,0.081562208,0,0,0,0,0,0,0,0,0,0,0,0
0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0.12431368,0.11188232,0.10069408,0.090624676,0,0,0,0,0,0,0,0,0,0,0,0
0.13812632,0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0.12431368,0.11188232,0.10069408,0,0,0,0,0,0,0,0,0,0,0,0
0.12431368,0.13812632,0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0.12431368,0.11188232,0,0,0,0,0,0,0,0,0,0,0,0
0.11188232,0.12431368,0.13812632,0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0.12431368,0,0,0,0,0,0,0,0,0,0,0,0
0.10069408,0.11188232,0.12431368,0.13812632,0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0.13812632,0,0,0,0,0,0,0,0,0,0,0,0
0.090624676,0.10069408,0.11188232,0.12431368,0.13812632,0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0.15347368,0,0,0,0,0,0,0,0,0,0,0,0
0.081562208,0.090624676,0.10069408,0.11188232,0.12431368,0.13812632,0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0.17052632,0,0,0,0,0,0,0,0,0,0,0,0
0.073405987,0.081562208,0.090624676,0.10069408,0.11188232,0.12431368,0.13812632,0.15347368,0.17052632,0.18947368,0.21052632,0.18947368,0,0,0,0,0,0,0,0,0,0,0,0


## Check shifts and the stationary covariance equation

In [6]:
matrix_checks = []
for phi in [-0.72, 0.0, TRUE_PARAMETERS['phi']]:
    p = Parameters(parameters.mu, phi, parameters.Q, parameters.H)
    m = augmented_matrices(p)
    x = np.arange(24, dtype=float)
    next_x = m['T']@x + m['R']@np.array([2.0,3.0])
    np.testing.assert_array_equal(next_x[1:12], x[:11])
    np.testing.assert_array_equal(next_x[13:], x[12:23])
    assert next_x[0] == phi*x[0]+2 and next_x[12] == 3
    residual = m['P0']-(m['T']@m['P0']@m['T'].T+m['R']@m['Q_aug']@m['R'].T)
    np.testing.assert_allclose(residual, 0, atol=1e-14)
    assert np.linalg.eigvalsh(m['P0']).min() > 0
    assert np.all(m['P0'][:12,12:] == 0)
    matrix_checks.append({'phi': phi, 'Max stationary residual': float(np.abs(residual).max()),
                          'Minimum prior eigenvalue': float(np.linalg.eigvalsh(m['P0']).min()),
                          'Shift relations': 'Passed'})
display(pd.DataFrame(matrix_checks))

,phi,Max stationary residual,Minimum prior eigenvalue,Shift relations
0,-0.72,3.469447e-18,0.01,Passed
1,0.00,0.000000e+00,0.01,Passed
2,0.90,2.775558e-17,0.01,Passed


All initial lags are correlated within the latent block and independent within the observation-shock block. The unconditional prior precedes the first annual observation; annual-only estimation cannot initialize from monthly observations. Carrying monthly observation shocks accounts for adjacent windows sharing eleven months. H_aug=0 removes extra annual noise, while the original H remains positive in the transition shocks and prior.